# Подготовка датасета MovieLens с явными рейтингами

Этот ноутбук собирает итоговый датасет для экспериментов по рекомендательным системам.  
В нём выполняется скачивание MovieLens 25M, добавление дополнительных признаков фильмов, фильтрация взаимодействий и **сплит по времени**.

На выходе получается набор `parquet` и `json` файлов, который дальше используется в ноутбуках с EDA, обучением моделей и сравнением результатов.

Что делает ноутбук:

- скачивает MovieLens 25M и нужные IMDb-таблицы;
- загружает рейтинги, фильмы, ссылки и MovieLens Genome;
- применяет k-core фильтрацию пользователей и фильмов;
- строит **сплит по времени** на train / validation / test;
- сохраняет все рейтинги в исходном численном виде, без бинаризации на этапе подготовки;
- выделяет warm-start, cold-user, cold-item и both-cold сценарии;
- строит дополнительные признаки фильмов из MovieLens, Genome и IMDb;
- сохраняет interaction-таблицы, item features, meta-таблицы и mapping-файлы.

Важно: положительные оценки используются только для задания границ validation/test окон.  
В сами validation/test таблицы попадают все рейтинги внутри соответствующих временных окон.


In [1]:
import gc
import os
import json
import math
import re
import shutil
import subprocess
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import TruncatedSVD

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)

# ------------------------------------------------------------
# Настройки
# ------------------------------------------------------------
SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

DEFAULT_WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("data")
WORK_DIR = Path(os.environ.get("WORK_DIR", str(DEFAULT_WORK_DIR)))
RAW_DIR = WORK_DIR / "raw_data"
IMDB_DIR = RAW_DIR / "imdb"
OUT_DIR = WORK_DIR / "processed_ml25m_cold_eval_all_ratings_v2"

RAW_DIR.mkdir(parents=True, exist_ok=True)
IMDB_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

ML_25M_ZIP_URL = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
IMDB_BASE_URL = "https://datasets.imdbws.com"

# Нужны только те файлы IMDb, которые реально используются в feature pipeline
IMDB_FILES = [
    "title.basics.tsv.gz",
    "title.crew.tsv.gz",
    "title.principals.tsv.gz",
]

# ---------- split ----------
POSITIVE_THRESHOLD = 4.0
RATING_WEIGHT_CENTER = 3.0

N_VAL_POS = 2
N_TEST_POS = 3
MIN_POS_PER_USER_FOR_SPLIT = N_VAL_POS + N_TEST_POS + 1

MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

# ---------- cold slices ----------
# Можно задавать либо долю (<1.0), либо абсолютное число (целое >= 1)
COLD_USER_HOLDOUT = 0.10
COLD_ITEM_HOLDOUT = 0.05

MIN_SUPPORT_INTERACTIONS_FOR_COLD_USER = 5
MIN_SUPPORT_POSITIVES_FOR_COLD_USER = 1

MIN_ITEM_TOTAL_INTERACTIONS_FOR_COLD = 20
MIN_ITEM_POSITIVE_INTERACTIONS_FOR_COLD = 10
MIN_ITEM_UNIQUE_USERS_FOR_COLD = 10

# ---------- item features ----------
GENOME_SVD_DIM = 64
PERSON_TOPK_THRESHOLDS = (50, 100, 250, 500)
PRINCIPAL_CATEGORIES = ("actor", "actress", "director", "writer")

# ---------- service / non-model columns ----------
NON_MODEL_ITEM_COLS = {
    "movieId",
    "title",
    "imdbId",
    "tmdbId",
    "tconst",
    "all_item_idx",
    "warm_item_idx",
    "cold_item_idx",
    "is_warm_item",
    "is_cold_item",
    "selected_cold_item",
    "item_split_group",
}


## 1. Скачивание исходных данных

Здесь скачиваются архив MovieLens 25M и IMDb-таблицы, которые нужны для построения дополнительных признаков фильмов.  
Для запуска этой части нужен доступ к интернету.


In [2]:
def download_file(url: str, out_path: Path, overwrite: bool = False):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and not overwrite:
        print(f"[skip] exists: {out_path}")
        return
    print(f"[download] {url} -> {out_path}")
    subprocess.run(["wget", "-nv", "-O", str(out_path), url], check=True)

def unzip_file(zip_path: Path, extract_dir: Path, overwrite: bool = False):
    extract_dir.mkdir(parents=True, exist_ok=True)
    marker_dir = extract_dir / "ml-25m"
    if marker_dir.exists() and not overwrite:
        print(f"[skip] already extracted: {marker_dir}")
        return
    print(f"[unzip] {zip_path} -> {extract_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

ml_zip_path = RAW_DIR / "ml-25m.zip"
download_file(ML_25M_ZIP_URL, ml_zip_path)
unzip_file(ml_zip_path, RAW_DIR)

for fname in IMDB_FILES:
    download_file(f"{IMDB_BASE_URL}/{fname}", IMDB_DIR / fname)

print("MovieLens files:")
for p in sorted((RAW_DIR / "ml-25m").glob("*")):
    print("  ", p.name)

print("\nIMDb files:")
for p in sorted(IMDB_DIR.glob("*")):
    print("  ", p.name)

[download] https://files.grouplens.org/datasets/movielens/ml-25m.zip -> /kaggle/working/raw_data/ml-25m.zip


2026-04-20 00:32:19 URL:https://files.grouplens.org/datasets/movielens/ml-25m.zip [261978986/261978986] -> "/kaggle/working/raw_data/ml-25m.zip" [1]


[unzip] /kaggle/working/raw_data/ml-25m.zip -> /kaggle/working/raw_data
[download] https://datasets.imdbws.com/title.basics.tsv.gz -> /kaggle/working/raw_data/imdb/title.basics.tsv.gz


2026-04-20 00:32:27 URL:https://datasets.imdbws.com/title.basics.tsv.gz [220589072/220589072] -> "/kaggle/working/raw_data/imdb/title.basics.tsv.gz" [1]


[download] https://datasets.imdbws.com/title.crew.tsv.gz -> /kaggle/working/raw_data/imdb/title.crew.tsv.gz


2026-04-20 00:32:28 URL:https://datasets.imdbws.com/title.crew.tsv.gz [81135049/81135049] -> "/kaggle/working/raw_data/imdb/title.crew.tsv.gz" [1]


[download] https://datasets.imdbws.com/title.principals.tsv.gz -> /kaggle/working/raw_data/imdb/title.principals.tsv.gz
MovieLens files:
   README.txt
   genome-scores.csv
   genome-tags.csv
   links.csv
   movies.csv
   ratings.csv
   tags.csv

IMDb files:
   title.basics.tsv.gz
   title.crew.tsv.gz
   title.principals.tsv.gz


2026-04-20 00:32:30 URL:https://datasets.imdbws.com/title.principals.tsv.gz [762386920/762386920] -> "/kaggle/working/raw_data/imdb/title.principals.tsv.gz" [1]


## 2. Загрузка исходных таблиц

Загружаются основные таблицы MovieLens: рейтинги, фильмы, ссылки и Genome scores.  
После этого данные ещё не разбиты на train/validation/test.


In [3]:
ML_ROOT = RAW_DIR / "ml-25m"

ratings = pd.read_csv(
    ML_ROOT / "ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"},
)
movies = pd.read_csv(
    ML_ROOT / "movies.csv",
    dtype={"movieId": "int32", "title": "string", "genres": "string"},
)
links = pd.read_csv(
    ML_ROOT / "links.csv",
    dtype={"movieId": "int32", "imdbId": "Int64", "tmdbId": "Int64"},
)
genome_scores = pd.read_csv(
    ML_ROOT / "genome-scores.csv",
    dtype={"movieId": "int32", "tagId": "int32", "relevance": "float32"},
)

print("ratings:", ratings.shape)
print("movies:", movies.shape)
print("links:", links.shape)
print("genome_scores:", genome_scores.shape)

ratings: (25000095, 4)
movies: (62423, 3)
links: (62423, 3)
genome_scores: (15584448, 3)


## 3. Служебные функции

В этом блоке собраны небольшие функции для сохранения файлов, вывода статистики, k-core фильтрации и работы с индексами.


In [4]:
def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def print_df_info(name: str, df: pd.DataFrame, max_cols: int = 40):
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print(df.dtypes.head(max_cols))
    display(df.head(3))

def sanitize_token(x: str) -> str:
    x = str(x).strip().lower()
    x = re.sub(r"[^a-z0-9_]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def extract_year_from_title(title: str):
    if pd.isna(title):
        return np.nan
    m = re.search(r"\((\d{4})\)\s*$", str(title))
    if m:
        return float(m.group(1))
    return np.nan

def resolve_holdout_size(n_candidates: int, value) -> int:
    if n_candidates <= 0:
        return 0
    if isinstance(value, float) and value < 1.0:
        n = int(math.floor(n_candidates * value))
        return max(1, min(n_candidates, n))
    return max(0, min(n_candidates, int(value)))

def iterative_k_core(
    interactions: pd.DataFrame,
    min_user_interactions: int = 5,
    min_item_interactions: int = 5,
) -> pd.DataFrame:
    df = interactions.copy()
    while True:
        n_before = len(df)

        user_counts = df["userId"].value_counts()
        keep_users = user_counts[user_counts >= min_user_interactions].index
        df = df[df["userId"].isin(keep_users)]

        item_counts = df["movieId"].value_counts()
        keep_items = item_counts[item_counts >= min_item_interactions].index
        df = df[df["movieId"].isin(keep_items)]

        if len(df) == n_before:
            break

    return df.reset_index(drop=True)

def attach_id_maps(
    df: pd.DataFrame,
    warm_user2idx: dict | None = None,
    warm_item2idx: dict | None = None,
    all_item2idx: dict | None = None,
    cold_user2idx: dict | None = None,
    cold_item2idx: dict | None = None,
) -> pd.DataFrame:
    out = df.copy()

    if warm_user2idx is not None:
        out["warm_user_idx"] = out["userId"].map(warm_user2idx).fillna(-1).astype("int32")
    if warm_item2idx is not None:
        out["warm_item_idx"] = out["movieId"].map(warm_item2idx).fillna(-1).astype("int32")
    if all_item2idx is not None:
        out["all_item_idx"] = out["movieId"].map(all_item2idx).fillna(-1).astype("int32")
    if cold_user2idx is not None:
        out["cold_user_idx"] = out["userId"].map(cold_user2idx).fillna(-1).astype("int32")
    if cold_item2idx is not None:
        out["cold_item_idx"] = out["movieId"].map(cold_item2idx).fillna(-1).astype("int32")

    return out

## 4. Функции для сплита по времени

Разбиение строится отдельно для каждого пользователя после сортировки его истории по времени.  
Границы validation и test задаются по последним положительным оценкам, но внутри самих окон сохраняются все рейтинги.


In [5]:
def build_multi_window_split(
    ratings_df: pd.DataFrame,
    positive_threshold: float = 4.0,
    n_val_pos: int = 2,
    n_test_pos: int = 3,
    min_pos_per_user: int = 6,
):
    """
    For each user:
      - sort all interactions by (timestamp, original_row_id)
      - find positive-anchor interactions by rating >= positive_threshold
      - last n_test_pos positive anchors define the start of the test window
      - previous n_val_pos positive anchors define the start of the val window
      - train history = all interactions before the first val anchor
      - val window = all interactions from the first val anchor up to the first test anchor
      - test window = all interactions from the first test anchor until the end

    This keeps the original temporal split logic, but stores *all* explicit ratings
    in train/val/test windows instead of only positive anchor interactions.
    """
    df = ratings_df.copy().reset_index(drop=False).rename(columns={"index": "_row_id"})
    df["is_positive"] = (df["rating"] >= positive_threshold).astype("uint8")
    df = df.sort_values(["userId", "timestamp", "_row_id"]).reset_index(drop=True)

    train_parts = []
    val_parts = []
    test_parts = []
    user_stats = []

    for user_id, g in df.groupby("userId", sort=False):
        g = g.sort_values(["timestamp", "_row_id"]).reset_index(drop=True)
        pos_positions = np.flatnonzero(g["is_positive"].to_numpy())

        if len(pos_positions) < min_pos_per_user:
            continue

        val_anchor_positions = pos_positions[-(n_val_pos + n_test_pos):-n_test_pos]
        test_anchor_positions = pos_positions[-n_test_pos:]

        if len(val_anchor_positions) != n_val_pos or len(test_anchor_positions) != n_test_pos:
            continue

        first_val_pos = int(val_anchor_positions[0])
        first_test_pos = int(test_anchor_positions[0])

        if first_val_pos <= 0 or first_test_pos <= first_val_pos:
            continue

        history = g.iloc[:first_val_pos].copy()
        val_window = g.iloc[first_val_pos:first_test_pos].copy()
        test_window = g.iloc[first_test_pos:].copy()

        train_pos_count = int(history["is_positive"].sum())
        if train_pos_count <= 0:
            continue
        if len(val_window) == 0 or len(test_window) == 0:
            continue

        train_parts.append(history[["userId", "movieId", "rating", "timestamp"]].copy())
        val_parts.append(val_window[["userId", "movieId", "rating", "timestamp"]].copy())
        test_parts.append(test_window[["userId", "movieId", "rating", "timestamp"]].copy())

        user_stats.append(
            {
                "userId": int(user_id),
                "n_interactions_total": int(len(g)),
                "n_positive_total": int(len(pos_positions)),
                "n_train_interactions": int(len(history)),
                "n_train_positive": int(train_pos_count),
                "n_val_interactions": int(len(val_window)),
                "n_test_interactions": int(len(test_window)),
                "n_val_positive_anchors": int(n_val_pos),
                "n_test_positive_anchors": int(n_test_pos),
                "n_val_positive_interactions": int(val_window["is_positive"].sum()),
                "n_test_positive_interactions": int(test_window["is_positive"].sum()),
                "first_val_timestamp": int(val_window["timestamp"].min()),
                "first_test_timestamp": int(test_window["timestamp"].min()),
                "last_test_timestamp": int(test_window["timestamp"].max()),
            }
        )

    if not train_parts:
        raise RuntimeError("No eligible users after split. Relax filters or reduce holdout sizes.")

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)
    user_stats_df = pd.DataFrame(user_stats)

    return train_df, val_df, test_df, user_stats_df

def select_cold_users(
    user_stats_df: pd.DataFrame,
    holdout,
    min_support_interactions: int,
    min_support_positives: int,
    seed: int = 42,
):
    candidates = user_stats_df[
        (user_stats_df["n_train_interactions"] >= min_support_interactions)
        & (user_stats_df["n_train_positive"] >= min_support_positives)
    ].copy()

    n_select = resolve_holdout_size(len(candidates), holdout)
    if n_select <= 0:
        return []

    rng_local = np.random.default_rng(seed)
    chosen = rng_local.choice(candidates["userId"].to_numpy(), size=n_select, replace=False)
    return sorted(pd.Series(chosen, dtype="int32").tolist())

def select_cold_items(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    warm_user_ids,
    holdout,
    min_total_interactions: int,
    min_positive_interactions: int,
    min_unique_users: int,
    seed: int = 42,
):
    warm_user_ids = set(pd.Series(warm_user_ids, dtype="int32").tolist())

    train_part = train_df[train_df["userId"].isin(warm_user_ids)][["userId", "movieId", "rating"]].copy()
    train_part["is_positive"] = (train_part["rating"] >= POSITIVE_THRESHOLD).astype("uint8")
    train_part = train_part[["userId", "movieId", "is_positive"]]

    val_part = val_df[val_df["userId"].isin(warm_user_ids)][["userId", "movieId", "rating"]].copy()
    val_part["is_positive"] = (val_part["rating"] >= POSITIVE_THRESHOLD).astype("uint8")
    val_part = val_part[["userId", "movieId", "is_positive"]]

    test_part = test_df[test_df["userId"].isin(warm_user_ids)][["userId", "movieId", "rating"]].copy()
    test_part["is_positive"] = (test_part["rating"] >= POSITIVE_THRESHOLD).astype("uint8")
    test_part = test_part[["userId", "movieId", "is_positive"]]

    full = pd.concat([train_part, val_part, test_part], ignore_index=True)
    if len(full) == 0:
        return [], pd.DataFrame(columns=["movieId", "n_interactions_total", "n_positive_total", "n_unique_users"])

    stats = (
        full.groupby("movieId")
        .agg(
            n_interactions_total=("userId", "size"),
            n_positive_total=("is_positive", "sum"),
            n_unique_users=("userId", "nunique"),
        )
        .reset_index()
    )

    eval_items = set(val_part["movieId"].tolist()) | set(test_part["movieId"].tolist())
    candidates = stats[
        (stats["n_interactions_total"] >= min_total_interactions)
        & (stats["n_positive_total"] >= min_positive_interactions)
        & (stats["n_unique_users"] >= min_unique_users)
        & (stats["movieId"].isin(eval_items))
    ].copy()

    n_select = resolve_holdout_size(len(candidates), holdout)
    if n_select <= 0:
        return [], stats

    rng_local = np.random.default_rng(seed)
    chosen = rng_local.choice(candidates["movieId"].to_numpy(), size=n_select, replace=False)
    return sorted(pd.Series(chosen, dtype="int32").tolist()), stats

def split_eval_interactions(df: pd.DataFrame, warm_user_ids, warm_item_ids):
    warm_user_ids = set(warm_user_ids)
    warm_item_ids = set(warm_item_ids)

    user_is_warm = df["userId"].isin(warm_user_ids)
    item_is_warm = df["movieId"].isin(warm_item_ids)

    warm = df[user_is_warm & item_is_warm].copy()
    cold_user = df[~user_is_warm & item_is_warm].copy()
    cold_item = df[user_is_warm & ~item_is_warm].copy()
    both_cold = df[~user_is_warm & ~item_is_warm].copy()

    return warm, cold_user, cold_item, both_cold

## 5. Признаки фильмов из MovieLens и Genome

В этом блоке строятся базовые признаки фильмов: год, десятилетие, жанры MovieLens и сжатое представление Genome-признаков через SVD.


In [6]:
def build_genre_features(movies_df: pd.DataFrame) -> pd.DataFrame:
    tmp = movies_df[["movieId", "genres"]].copy()
    tmp["genres"] = tmp["genres"].fillna("(no genres listed)")
    tmp["genre_list"] = tmp["genres"].str.split("|")
    tmp["genre_count"] = (
        tmp["genre_list"]
        .apply(lambda xs: sum(bool(str(x).strip()) for x in xs) if isinstance(xs, list) else 0)
        .astype("int16")
    )

    exploded = tmp[["movieId", "genre_list"]].explode("genre_list").rename(columns={"genre_list": "genre"})
    exploded["genre"] = exploded["genre"].fillna("(no genres listed)").map(sanitize_token)

    genre_mh = pd.crosstab(exploded["movieId"], exploded["genre"]).astype("uint8")
    genre_mh.columns = [f"genre__{c}" for c in genre_mh.columns]
    genre_mh = genre_mh.reset_index()

    out = tmp[["movieId", "genre_count"]].drop_duplicates("movieId").merge(genre_mh, on="movieId", how="left")
    for c in out.columns:
        if c.startswith("genre__"):
            out[c] = out[c].fillna(0).astype("uint8")
    out["genre_count"] = out["genre_count"].fillna(0).astype("int16")
    return out

def build_genome_svd_features(
    genome_scores_df: pd.DataFrame,
    movie_ids,
    n_components: int = 64,
) -> pd.DataFrame:
    movie_ids = pd.Index(sorted(pd.unique(pd.Series(movie_ids, dtype="int32"))))
    gs = genome_scores_df[genome_scores_df["movieId"].isin(movie_ids)].copy()

    out = pd.DataFrame({"movieId": movie_ids.astype("int32")})
    available_movie_ids = set(gs["movieId"].unique().tolist())
    out["genome_missing"] = (~out["movieId"].isin(available_movie_ids)).astype("uint8")

    if len(gs) == 0:
        return out

    uniq_movies = np.sort(gs["movieId"].unique())
    uniq_tags = np.sort(gs["tagId"].unique())

    movie_cat = pd.Categorical(gs["movieId"], categories=uniq_movies)
    tag_cat = pd.Categorical(gs["tagId"], categories=uniq_tags)

    X = sparse.csr_matrix(
        (
            gs["relevance"].astype("float32").values,
            (movie_cat.codes, tag_cat.codes),
        ),
        shape=(len(uniq_movies), len(uniq_tags)),
        dtype=np.float32,
    )

    max_possible = min(X.shape[0] - 1, X.shape[1] - 1)
    if max_possible <= 0:
        return out

    n_comp = min(n_components, max_possible)
    print(f"Genome matrix shape={X.shape}, TruncatedSVD n_components={n_comp}")

    svd = TruncatedSVD(
        n_components=n_comp,
        algorithm="randomized",
        random_state=SEED,
    )
    Z = svd.fit_transform(X).astype("float32")

    svd_cols = [f"genome_svd_{i:03d}" for i in range(n_comp)]
    genome_feat = pd.DataFrame(Z, columns=svd_cols)
    genome_feat.insert(0, "movieId", uniq_movies.astype("int32"))

    out = out.merge(genome_feat, on="movieId", how="left")
    for c in svd_cols:
        out[c] = out[c].fillna(0).astype("float32")

    return out

def build_base_item_features(
    movies_df: pd.DataFrame,
    links_df: pd.DataFrame,
    genome_scores_df: pd.DataFrame,
    item_movie_ids,
    genome_svd_dim: int = 64,
) -> pd.DataFrame:
    item_movie_ids = pd.Index(sorted(pd.unique(pd.Series(item_movie_ids, dtype="int32"))))
    items_df = pd.DataFrame({"movieId": item_movie_ids.astype("int32")})

    mv = movies_df[movies_df["movieId"].isin(item_movie_ids)].copy()
    mv["year"] = mv["title"].map(extract_year_from_title).astype("float32")
    mv["year_missing"] = mv["year"].isna().astype("uint8")
    mv["decade"] = ((mv["year"] // 10) * 10).fillna(0).astype("float32")

    genre_feat = build_genre_features(mv)

    lk = links_df[links_df["movieId"].isin(item_movie_ids)][["movieId", "imdbId", "tmdbId"]].drop_duplicates("movieId")

    genome_feat = build_genome_svd_features(
        genome_scores_df=genome_scores_df,
        movie_ids=item_movie_ids,
        n_components=genome_svd_dim,
    )

    item_features = (
        items_df
        .merge(mv[["movieId", "title", "year", "year_missing", "decade"]], on="movieId", how="left")
        .merge(lk, on="movieId", how="left")
        .merge(genre_feat, on="movieId", how="left")
        .merge(genome_feat, on="movieId", how="left")
    )

    item_features["year"] = item_features["year"].fillna(0).astype("float32")
    item_features["year_missing"] = item_features["year_missing"].fillna(1).astype("uint8")
    item_features["decade"] = item_features["decade"].fillna(0).astype("float32")
    item_features["genre_count"] = item_features["genre_count"].fillna(0).astype("int16")

    for c in item_features.columns:
        if c.startswith("genre__"):
            item_features[c] = item_features[c].fillna(0).astype("uint8")
        elif c.startswith("genome_svd_"):
            item_features[c] = item_features[c].fillna(0).astype("float32")

    item_features["genome_missing"] = item_features["genome_missing"].fillna(1).astype("uint8")
    return item_features

## 6. IMDb-признаки

Здесь добавляются признаки из IMDb: тип тайтла, длительность, год, жанры, а также агрегаты по актёрам, режиссёрам, сценаристам и основным участникам.


In [7]:
TITLE_BASICS_PATH = IMDB_DIR / "title.basics.tsv.gz"
TITLE_CREW_PATH = IMDB_DIR / "title.crew.tsv.gz"
TITLE_PRINCIPALS_PATH = IMDB_DIR / "title.principals.tsv.gz"

def imdb_numeric_to_tconst(imdb_id_series: pd.Series) -> pd.Series:
    x = pd.to_numeric(imdb_id_series, errors="coerce").astype("Int64")
    out = pd.Series(pd.NA, index=imdb_id_series.index, dtype="string")
    mask = x.notna()
    out.loc[mask] = "tt" + x.loc[mask].astype(str).str.zfill(7)
    return out

def read_tsv_in_chunks(path: Path, usecols, chunksize=1_000_000):
    return pd.read_csv(
        path,
        sep="\t",
        compression="gzip",
        usecols=usecols,
        dtype="string",
        na_values="\\N",
        keep_default_na=True,
        low_memory=False,
        chunksize=chunksize,
    )

def load_filtered_tsv(path: Path, usecols, tconst_set=None, extra_filter_fn=None, chunksize=1_000_000):
    parts = []
    for chunk in read_tsv_in_chunks(path, usecols=usecols, chunksize=chunksize):
        if tconst_set is not None and "tconst" in chunk.columns:
            chunk = chunk[chunk["tconst"].isin(tconst_set)]
        if extra_filter_fn is not None:
            chunk = extra_filter_fn(chunk)
        if len(chunk):
            parts.append(chunk)
    if not parts:
        return pd.DataFrame(columns=usecols)
    return pd.concat(parts, ignore_index=True)

def multi_hot_from_delimited_column(df, id_col, value_col, prefix, sep=","):
    tmp = df[[id_col, value_col]].copy()
    tmp[value_col] = tmp[value_col].fillna("")
    tmp[value_col] = tmp[value_col].str.split(sep)
    tmp = tmp.explode(value_col)
    tmp[value_col] = tmp[value_col].astype("string").str.strip()
    tmp = tmp[tmp[value_col].notna() & (tmp[value_col] != "")]
    if len(tmp) == 0:
        return pd.DataFrame({id_col: df[id_col].drop_duplicates().tolist()})
    mh = pd.crosstab(tmp[id_col], tmp[value_col]).astype("uint8")
    mh.columns = [f"{prefix}__{sanitize_token(c)}" for c in mh.columns]
    return mh.reset_index()

def explode_csv_people(df: pd.DataFrame, source_col: str, target_col: str = "nconst") -> pd.DataFrame:
    tmp = df[["tconst", source_col]].copy()
    tmp[source_col] = tmp[source_col].fillna("").str.split(",")
    tmp = tmp.explode(source_col).rename(columns={source_col: target_col})
    tmp[target_col] = tmp[target_col].astype("string").str.strip()
    tmp = tmp[tmp[target_col].notna() & (tmp[target_col] != "")]
    tmp = tmp.drop_duplicates(["tconst", target_col]).reset_index(drop=True)
    return tmp

def build_person_rank_features(base_tconsts, pairs_df: pd.DataFrame, prefix: str, thresholds=(50, 100, 250, 500)) -> pd.DataFrame:
    base_tconsts = pd.Index(sorted(pd.unique(pd.Series(list(base_tconsts), dtype="string"))))
    out = pd.DataFrame({"tconst": base_tconsts})

    count_cols = [f"{prefix}_top{k}_count" for k in thresholds]

    if pairs_df is None or len(pairs_df) == 0:
        for c in count_cols:
            out[c] = np.uint16(0)
        out[f"{prefix}_popularity_max"] = np.float32(0)
        out[f"{prefix}_popularity_mean"] = np.float32(0)
        out[f"{prefix}_popularity_sum"] = np.float32(0)
        return out

    pairs = pairs_df[["tconst", "nconst"]].dropna().drop_duplicates()
    if len(pairs) == 0:
        for c in count_cols:
            out[c] = np.uint16(0)
        out[f"{prefix}_popularity_max"] = np.float32(0)
        out[f"{prefix}_popularity_mean"] = np.float32(0)
        out[f"{prefix}_popularity_sum"] = np.float32(0)
        return out

    popularity = (
        pairs.groupby("nconst")["tconst"]
        .nunique()
        .sort_values(ascending=False)
        .rename("popularity")
        .reset_index()
    )
    popularity["pop_rank"] = np.arange(1, len(popularity) + 1, dtype=np.int32)

    tmp = pairs.merge(popularity, on="nconst", how="left")

    pop_agg = (
        tmp.groupby("tconst")["popularity"]
        .agg(["max", "mean", "sum"])
        .reset_index()
        .rename(
            columns={
                "max": f"{prefix}_popularity_max",
                "mean": f"{prefix}_popularity_mean",
                "sum": f"{prefix}_popularity_sum",
            }
        )
    )
    out = out.merge(pop_agg, on="tconst", how="left")

    for k in thresholds:
        cnt = (
            tmp[tmp["pop_rank"] <= k]
            .groupby("tconst")["nconst"]
            .nunique()
            .rename(f"{prefix}_top{k}_count")
            .reset_index()
        )
        out = out.merge(cnt, on="tconst", how="left")

    for c in count_cols:
        out[c] = out[c].fillna(0).astype("uint16")
    out[f"{prefix}_popularity_max"] = out[f"{prefix}_popularity_max"].fillna(0).astype("float32")
    out[f"{prefix}_popularity_mean"] = out[f"{prefix}_popularity_mean"].fillna(0).astype("float32")
    out[f"{prefix}_popularity_sum"] = out[f"{prefix}_popularity_sum"].fillna(0).astype("float32")

    return out

def build_imdb_item_features(
    links_df: pd.DataFrame,
    catalog_movie_ids,
    topk_thresholds=(50, 100, 250, 500),
):
    catalog_movie_ids = pd.Index(sorted(pd.unique(pd.Series(catalog_movie_ids, dtype="int32"))))
    all_movies = pd.DataFrame({"movieId": catalog_movie_ids.astype("int32")})

    link = links_df[links_df["movieId"].isin(catalog_movie_ids)][["movieId", "imdbId"]].copy()
    link["tconst"] = imdb_numeric_to_tconst(link["imdbId"])
    link = link.dropna(subset=["tconst"]).drop_duplicates("movieId")
    tconst_set = set(link["tconst"].tolist())

    imdb_feat = all_movies.merge(link, on="movieId", how="left")
    imdb_feat["has_imdb_match"] = imdb_feat["tconst"].notna().astype("uint8")

    basics = load_filtered_tsv(
        TITLE_BASICS_PATH,
        usecols=["tconst", "titleType", "isAdult", "startYear", "runtimeMinutes", "genres"],
        tconst_set=tconst_set,
        chunksize=500_000,
    )

    if len(basics):
        basics["isAdult"] = pd.to_numeric(basics["isAdult"], errors="coerce").fillna(0).astype("int8")
        basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce").fillna(0).astype("float32")
        basics["runtimeMinutes"] = pd.to_numeric(basics["runtimeMinutes"], errors="coerce").astype("float32")
        basics["runtime_missing"] = basics["runtimeMinutes"].isna().astype("uint8")
        basics["runtimeMinutes"] = basics["runtimeMinutes"].fillna(0).astype("float32")
        basics["runtimeMinutes_log1p"] = np.log1p(basics["runtimeMinutes"].clip(lower=0)).astype("float32")

        title_type_dummies = pd.get_dummies(
            basics["titleType"].fillna("unknown"),
            prefix="imdb_titleType",
            dtype="uint8",
        )

        basics_num = basics[
            ["tconst", "isAdult", "startYear", "runtimeMinutes", "runtimeMinutes_log1p", "runtime_missing"]
        ].drop_duplicates("tconst")

        imdb_genre_feat = multi_hot_from_delimited_column(
            basics[["tconst", "genres"]],
            id_col="tconst",
            value_col="genres",
            prefix="imdb_genre",
            sep=",",
        )

        basics_type_feat = (
            pd.concat([basics[["tconst"]].reset_index(drop=True), title_type_dummies.reset_index(drop=True)], axis=1)
            .groupby("tconst", as_index=False)
            .max()
        )

        basics_feat = basics_type_feat.merge(basics_num, on="tconst", how="left")
        basics_feat = basics_feat.merge(imdb_genre_feat, on="tconst", how="left")
        imdb_feat = imdb_feat.merge(basics_feat, on="tconst", how="left")

    del basics
    gc.collect()

    crew = load_filtered_tsv(
        TITLE_CREW_PATH,
        usecols=["tconst", "directors", "writers"],
        tconst_set=tconst_set,
        chunksize=500_000,
    )

    director_pairs = pd.DataFrame(columns=["tconst", "nconst"])
    if len(crew):
        def count_csv_people(s):
            if pd.isna(s) or s == "":
                return 0
            return len([x for x in str(s).split(",") if x])

        crew["n_directors"] = crew["directors"].map(count_csv_people).astype("int16")
        crew["n_writers"] = crew["writers"].map(count_csv_people).astype("int16")

        crew_feat = crew[["tconst", "n_directors", "n_writers"]].drop_duplicates("tconst")
        imdb_feat = imdb_feat.merge(crew_feat, on="tconst", how="left")

        director_pairs = explode_csv_people(crew, "directors")

    principals_all = load_filtered_tsv(
        TITLE_PRINCIPALS_PATH,
        usecols=["tconst", "nconst", "category"],
        tconst_set=tconst_set,
        chunksize=1_000_000,
    )

    if len(principals_all):
        n_principals = (
            principals_all.groupby("tconst")
            .size()
            .astype("int16")
            .rename("n_principals")
            .reset_index()
        )
        imdb_feat = imdb_feat.merge(n_principals, on="tconst", how="left")

        principals = principals_all[principals_all["category"].isin(PRINCIPAL_CATEGORIES)].copy()
        if len(principals):
            pc = pd.crosstab(principals["tconst"], principals["category"]).astype("uint16").reset_index()
            imdb_feat = imdb_feat.merge(pc, on="tconst", how="left")

        cast_pairs = principals_all[principals_all["category"].isin(["actor", "actress"])][["tconst", "nconst"]].drop_duplicates()

        if len(director_pairs) == 0:
            director_pairs = principals_all[principals_all["category"] == "director"][["tconst", "nconst"]].drop_duplicates()

        cast_rank_feat = build_person_rank_features(tconst_set, cast_pairs, prefix="cast", thresholds=topk_thresholds)
        dir_rank_feat = build_person_rank_features(tconst_set, director_pairs, prefix="director", thresholds=topk_thresholds)

        imdb_feat = imdb_feat.merge(cast_rank_feat, on="tconst", how="left")
        imdb_feat = imdb_feat.merge(dir_rank_feat, on="tconst", how="left")

    for c in ["n_principals", "n_directors", "n_writers", "actor", "actress", "director", "writer"]:
        if c in imdb_feat.columns:
            imdb_feat[c] = imdb_feat[c].fillna(0)

    imdb_feat["n_principals"] = imdb_feat.get("n_principals", 0)
    imdb_feat["n_directors"] = imdb_feat.get("n_directors", 0)
    imdb_feat["n_writers"] = imdb_feat.get("n_writers", 0)
    imdb_feat["n_actor"] = imdb_feat.get("actor", 0)
    imdb_feat["n_actress"] = imdb_feat.get("actress", 0)
    imdb_feat["n_cast_total"] = (imdb_feat["n_actor"] + imdb_feat["n_actress"]).astype("int16")

    distinct_cols = [c for c in ["actor", "actress", "director", "writer"] if c in imdb_feat.columns]
    if distinct_cols:
        imdb_feat["n_distinct_principal_categories"] = (imdb_feat[distinct_cols].fillna(0) > 0).sum(axis=1).astype("int16")
    else:
        imdb_feat["n_distinct_principal_categories"] = np.int16(0)

    imdb_feat["n_principals"] = pd.to_numeric(imdb_feat["n_principals"], errors="coerce").fillna(0).astype("int16")
    imdb_feat["n_directors"] = pd.to_numeric(imdb_feat["n_directors"], errors="coerce").fillna(0).astype("int16")
    imdb_feat["n_writers"] = pd.to_numeric(imdb_feat["n_writers"], errors="coerce").fillna(0).astype("int16")
    imdb_feat["n_actor"] = pd.to_numeric(imdb_feat["n_actor"], errors="coerce").fillna(0).astype("int16")
    imdb_feat["n_actress"] = pd.to_numeric(imdb_feat["n_actress"], errors="coerce").fillna(0).astype("int16")
    imdb_feat["n_cast_total"] = pd.to_numeric(imdb_feat["n_cast_total"], errors="coerce").fillna(0).astype("int16")
    imdb_feat["n_distinct_principal_categories"] = pd.to_numeric(
        imdb_feat["n_distinct_principal_categories"], errors="coerce"
    ).fillna(0).astype("int16")

    for k in topk_thresholds:
        cast_count_col = f"cast_top{k}_count"
        dir_count_col = f"director_top{k}_count"

        if cast_count_col not in imdb_feat.columns:
            imdb_feat[cast_count_col] = np.uint16(0)
        if dir_count_col not in imdb_feat.columns:
            imdb_feat[dir_count_col] = np.uint16(0)

        imdb_feat[cast_count_col] = pd.to_numeric(imdb_feat[cast_count_col], errors="coerce").fillna(0).astype("uint16")
        imdb_feat[dir_count_col] = pd.to_numeric(imdb_feat[dir_count_col], errors="coerce").fillna(0).astype("uint16")

        imdb_feat[f"cast_top{k}_share"] = np.where(
            imdb_feat["n_cast_total"] > 0,
            imdb_feat[cast_count_col] / imdb_feat["n_cast_total"],
            0.0,
        ).astype("float32")

        imdb_feat[f"director_top{k}_share"] = np.where(
            imdb_feat["n_directors"] > 0,
            imdb_feat[dir_count_col] / imdb_feat["n_directors"],
            0.0,
        ).astype("float32")

    for c in [
        "cast_popularity_max",
        "cast_popularity_mean",
        "cast_popularity_sum",
        "director_popularity_max",
        "director_popularity_mean",
        "director_popularity_sum",
        "isAdult",
        "startYear",
        "runtimeMinutes",
        "runtimeMinutes_log1p",
        "runtime_missing",
    ]:
        if c not in imdb_feat.columns:
            imdb_feat[c] = 0

    imdb_feat["isAdult"] = pd.to_numeric(imdb_feat["isAdult"], errors="coerce").fillna(0).astype("int8")
    imdb_feat["startYear"] = pd.to_numeric(imdb_feat["startYear"], errors="coerce").fillna(0).astype("float32")
    imdb_feat["runtimeMinutes"] = pd.to_numeric(imdb_feat["runtimeMinutes"], errors="coerce").fillna(0).astype("float32")
    imdb_feat["runtimeMinutes_log1p"] = pd.to_numeric(imdb_feat["runtimeMinutes_log1p"], errors="coerce").fillna(0).astype("float32")
    imdb_feat["runtime_missing"] = pd.to_numeric(imdb_feat["runtime_missing"], errors="coerce").fillna(1).astype("uint8")

    for c in imdb_feat.columns:
        if c in {"movieId", "imdbId", "tconst"}:
            continue
        if pd.api.types.is_numeric_dtype(imdb_feat[c]):
            imdb_feat[c] = imdb_feat[c].fillna(0)
        elif imdb_feat[c].dtype == "bool":
            imdb_feat[c] = imdb_feat[c].astype("uint8")

    imdb_feat = imdb_feat.drop(columns=["actor", "actress", "director", "writer"], errors="ignore")
    return imdb_feat

## 7. Запуск pipeline: фильтрация, сплит по времени и cold-сценарии

В этой части применяется k-core фильтрация, выполняется сплит по времени и отдельно выделяются cold-user / cold-item группы.  
Итоговый `train_warm` используется как основной обучающий граф.


In [8]:
print("Raw ratings:", ratings.shape)

ratings_k = iterative_k_core(
    ratings,
    min_user_interactions=MIN_USER_INTERACTIONS,
    min_item_interactions=MIN_ITEM_INTERACTIONS,
)

print("After k-core:", ratings_k.shape)
print("Unique users after k-core:", ratings_k["userId"].nunique())
print("Unique items after k-core:", ratings_k["movieId"].nunique())

train_base, val_base, test_base, user_split_stats = build_multi_window_split(
    ratings_df=ratings_k,
    positive_threshold=POSITIVE_THRESHOLD,
    n_val_pos=N_VAL_POS,
    n_test_pos=N_TEST_POS,
    min_pos_per_user=MIN_POS_PER_USER_FOR_SPLIT,
)

print_df_info("train_base", train_base)
print_df_info("val_base", val_base)
print_df_info("test_base", test_base)
print_df_info("user_split_stats", user_split_stats)

eligible_user_ids = sorted(user_split_stats["userId"].unique().tolist())
eligible_item_ids = sorted(set(train_base["movieId"].tolist()) | set(val_base["movieId"].tolist()) | set(test_base["movieId"].tolist()))

print("Eligible users:", len(eligible_user_ids))
print("Eligible items:", len(eligible_item_ids))

Raw ratings: (25000095, 4)
After k-core: (24945870, 4)
Unique users after k-core: 162541
Unique items after k-core: 32720

=== train_base ===
shape: (22998019, 4)
userId         int32
movieId        int32
rating       float32
timestamp      int64
dtype: object


,userId,movieId,rating,timestamp
0,1,5952,4.0,1147868053
1,1,2012,2.5,1147868068
2,1,2011,2.5,1147868079



=== val_base ===
shape: (710219, 4)
userId         int32
movieId        int32
rating       float32
timestamp      int64
dtype: object


,userId,movieId,rating,timestamp
0,1,8327,5.0,1147879375
1,1,32591,5.0,1147879538
2,1,5269,0.5,1147879571



=== test_base ===
shape: (1156112, 4)
userId         int32
movieId        int32
rating       float32
timestamp      int64
dtype: object


,userId,movieId,rating,timestamp
0,1,3569,5.0,1147879603
1,1,27193,3.0,1147879774
2,1,5684,2.0,1147879797



=== user_split_stats ===
shape: (159944, 14)
userId                          int64
n_interactions_total            int64
n_positive_total                int64
n_train_interactions            int64
n_train_positive                int64
n_val_interactions              int64
n_test_interactions             int64
n_val_positive_anchors          int64
n_test_positive_anchors         int64
n_val_positive_interactions     int64
n_test_positive_interactions    int64
first_val_timestamp             int64
first_test_timestamp            int64
last_test_timestamp             int64
dtype: object


,userId,n_interactions_total,n_positive_total,n_train_interactions,n_train_positive,n_val_interactions,n_test_interactions,n_val_positive_anchors,n_test_positive_anchors,n_val_positive_interactions,n_test_positive_interactions,first_val_timestamp,first_test_timestamp,last_test_timestamp
0,1,70,39,61,34,3,6,2,3,2,3,1147879375,1147879603,1147880055
1,2,184,113,175,108,5,4,2,3,2,3,1141417831,1141417937,1141418008
2,3,656,357,643,352,7,6,2,3,2,3,1566091441,1566091612,1566091883


Eligible users: 159944
Eligible items: 32720


In [9]:
sampled_cold_user_ids = set(
    select_cold_users(
        user_stats_df=user_split_stats,
        holdout=COLD_USER_HOLDOUT,
        min_support_interactions=MIN_SUPPORT_INTERACTIONS_FOR_COLD_USER,
        min_support_positives=MIN_SUPPORT_POSITIVES_FOR_COLD_USER,
        seed=SEED,
    )
)
sampled_warm_user_ids = sorted(set(eligible_user_ids) - sampled_cold_user_ids)

print("Sampled cold users:", len(sampled_cold_user_ids))
print("Remaining users for warm train candidate:", len(sampled_warm_user_ids))

sampled_cold_item_ids, candidate_item_stats = select_cold_items(
    train_df=train_base,
    val_df=val_base,
    test_df=test_base,
    warm_user_ids=sampled_warm_user_ids,
    holdout=COLD_ITEM_HOLDOUT,
    min_total_interactions=MIN_ITEM_TOTAL_INTERACTIONS_FOR_COLD,
    min_positive_interactions=MIN_ITEM_POSITIVE_INTERACTIONS_FOR_COLD,
    min_unique_users=MIN_ITEM_UNIQUE_USERS_FOR_COLD,
    seed=SEED,
)
sampled_cold_item_ids = set(sampled_cold_item_ids)
print("Sampled cold items:", len(sampled_cold_item_ids))

train_warm_provisional = train_base[
    (~train_base["userId"].isin(sampled_cold_user_ids))
    & (~train_base["movieId"].isin(sampled_cold_item_ids))
].copy()

users_with_warm_history = set(train_warm_provisional["userId"].unique().tolist())
auto_cold_user_ids = set(sampled_warm_user_ids) - users_with_warm_history

cold_user_ids = sampled_cold_user_ids | auto_cold_user_ids
warm_user_ids = sorted(set(eligible_user_ids) - cold_user_ids)

print("Auto-cold users (lost all warm history):", len(auto_cold_user_ids))
print("Final cold users:", len(cold_user_ids))
print("Final warm users:", len(warm_user_ids))

train_warm = train_base[
    (train_base["userId"].isin(warm_user_ids))
    & (~train_base["movieId"].isin(sampled_cold_item_ids))
].copy()

warm_item_ids = sorted(train_warm["movieId"].unique().tolist())
cold_item_ids = sorted(set(eligible_item_ids) - set(warm_item_ids))

print("Final warm train interactions:", train_warm.shape)
print("Final warm items:", len(warm_item_ids))
print("Final cold items:", len(cold_item_ids))

Sampled cold users: 15934
Remaining users for warm train candidate: 144010
Sampled cold items: 718
Auto-cold users (lost all warm history): 1
Final cold users: 15935
Final warm users: 144009
Final warm train interactions: (19574307, 4)
Final warm items: 31999
Final cold items: 721


In [10]:
warm_val_interactions, cold_user_val_interactions, cold_item_val_interactions, both_cold_val_interactions = split_eval_interactions(
    val_base, warm_user_ids=warm_user_ids, warm_item_ids=warm_item_ids
)
warm_test_interactions, cold_user_test_interactions, cold_item_test_interactions, both_cold_test_interactions = split_eval_interactions(
    test_base, warm_user_ids=warm_user_ids, warm_item_ids=warm_item_ids
)

cold_user_support_all = train_base[train_base["userId"].isin(cold_user_ids)].copy()
cold_user_support = cold_user_support_all[cold_user_support_all["movieId"].isin(warm_item_ids)].copy()

print_df_info("train_warm", train_warm)

print("warm_val_interactions:", warm_val_interactions.shape)
print("cold_user_val_interactions:", cold_user_val_interactions.shape)
print("cold_item_val_interactions:", cold_item_val_interactions.shape)
print("both_cold_val_interactions:", both_cold_val_interactions.shape)

print("warm_test_interactions:", warm_test_interactions.shape)
print("cold_user_test_interactions:", cold_user_test_interactions.shape)
print("cold_item_test_interactions:", cold_item_test_interactions.shape)
print("both_cold_test_interactions:", both_cold_test_interactions.shape)

print("cold_user_support_all:", cold_user_support_all.shape)
print("cold_user_support:", cold_user_support.shape)


=== train_warm ===
shape: (19574307, 4)
userId         int32
movieId        int32
rating       float32
timestamp      int64
dtype: object


,userId,movieId,rating,timestamp
61,2,2797,1.0,1141415509
62,2,5952,5.0,1141415528
63,2,1080,1.0,1141415532


warm_val_interactions: (604994, 4)
cold_user_val_interactions: (66243, 4)
cold_item_val_interactions: (35162, 4)
both_cold_val_interactions: (3820, 4)
warm_test_interactions: (987722, 4)
cold_user_test_interactions: (106919, 4)
cold_item_test_interactions: (55411, 4)
both_cold_test_interactions: (6060, 4)
cold_user_support_all: (2295457, 4)
cold_user_support: (2170334, 4)


## 8. Индексы пользователей, фильмов и meta-таблицы

Для моделей нужны компактные индексы пользователей и фильмов.  
Здесь создаются mapping-файлы и служебные meta-таблицы для warm/cold сценариев.


In [11]:
warm_user2idx = {int(uid): int(i) for i, uid in enumerate(sorted(train_warm["userId"].unique().tolist()))}
warm_item2idx = {int(mid): int(i) for i, mid in enumerate(warm_item_ids)}
all_item2idx = {int(mid): int(i) for i, mid in enumerate(sorted(eligible_item_ids))}
cold_user2idx = {int(uid): int(i) for i, uid in enumerate(sorted(cold_user_ids))}
cold_item2idx = {int(mid): int(i) for i, mid in enumerate(sorted(cold_item_ids))}

train_warm = attach_id_maps(
    train_warm,
    warm_user2idx=warm_user2idx,
    warm_item2idx=warm_item2idx,
    all_item2idx=all_item2idx,
)

warm_val_interactions = attach_id_maps(
    warm_val_interactions,
    warm_user2idx=warm_user2idx,
    warm_item2idx=warm_item2idx,
    all_item2idx=all_item2idx,
)
warm_test_interactions = attach_id_maps(
    warm_test_interactions,
    warm_user2idx=warm_user2idx,
    warm_item2idx=warm_item2idx,
    all_item2idx=all_item2idx,
)

cold_user_support_all = attach_id_maps(
    cold_user_support_all,
    warm_item2idx=warm_item2idx,
    all_item2idx=all_item2idx,
    cold_user2idx=cold_user2idx,
    cold_item2idx=cold_item2idx,
)
cold_user_support = attach_id_maps(
    cold_user_support,
    warm_item2idx=warm_item2idx,
    all_item2idx=all_item2idx,
    cold_user2idx=cold_user2idx,
)

cold_user_val_interactions = attach_id_maps(
    cold_user_val_interactions,
    warm_item2idx=warm_item2idx,
    all_item2idx=all_item2idx,
    cold_user2idx=cold_user2idx,
)
cold_user_test_interactions = attach_id_maps(
    cold_user_test_interactions,
    warm_item2idx=warm_item2idx,
    all_item2idx=all_item2idx,
    cold_user2idx=cold_user2idx,
)

cold_item_val_interactions = attach_id_maps(
    cold_item_val_interactions,
    warm_user2idx=warm_user2idx,
    all_item2idx=all_item2idx,
    cold_item2idx=cold_item2idx,
)
cold_item_test_interactions = attach_id_maps(
    cold_item_test_interactions,
    warm_user2idx=warm_user2idx,
    all_item2idx=all_item2idx,
    cold_item2idx=cold_item2idx,
)

both_cold_val_interactions = attach_id_maps(
    both_cold_val_interactions,
    all_item2idx=all_item2idx,
    cold_user2idx=cold_user2idx,
    cold_item2idx=cold_item2idx,
)
both_cold_test_interactions = attach_id_maps(
    both_cold_test_interactions,
    all_item2idx=all_item2idx,
    cold_user2idx=cold_user2idx,
    cold_item2idx=cold_item2idx,
)

def keep_raw_interaction_columns(df: pd.DataFrame) -> pd.DataFrame:
    preferred_cols = [
        "userId", "movieId", "rating", "timestamp",
        "warm_user_idx", "warm_item_idx", "all_item_idx",
        "cold_user_idx", "cold_item_idx",
    ]
    cols = [c for c in preferred_cols if c in df.columns]
    return df[cols].copy()

train_warm = keep_raw_interaction_columns(train_warm)
warm_val_interactions = keep_raw_interaction_columns(warm_val_interactions)
warm_test_interactions = keep_raw_interaction_columns(warm_test_interactions)
cold_user_support_all = keep_raw_interaction_columns(cold_user_support_all)
cold_user_support = keep_raw_interaction_columns(cold_user_support)
cold_user_val_interactions = keep_raw_interaction_columns(cold_user_val_interactions)
cold_user_test_interactions = keep_raw_interaction_columns(cold_user_test_interactions)
cold_item_val_interactions = keep_raw_interaction_columns(cold_item_val_interactions)
cold_item_test_interactions = keep_raw_interaction_columns(cold_item_test_interactions)
both_cold_val_interactions = keep_raw_interaction_columns(both_cold_val_interactions)
both_cold_test_interactions = keep_raw_interaction_columns(both_cold_test_interactions)

user_meta = user_split_stats.copy()
user_meta["is_cold_user"] = user_meta["userId"].isin(cold_user_ids).astype("uint8")
user_meta["is_warm_user"] = (1 - user_meta["is_cold_user"]).astype("uint8")
user_meta["warm_user_idx"] = user_meta["userId"].map(warm_user2idx).fillna(-1).astype("int32")
user_meta["cold_user_idx"] = user_meta["userId"].map(cold_user2idx).fillna(-1).astype("int32")
user_meta["user_split_group"] = np.where(user_meta["is_cold_user"] == 1, "cold", "warm")

def add_target_counts(meta_df, df, prefix):
    cnt = df.groupby("userId").size().rename(prefix).reset_index()
    return meta_df.merge(cnt, on="userId", how="left")

for prefix, df in [
    ("n_warm_val_interactions", warm_val_interactions),
    ("n_warm_test_interactions", warm_test_interactions),
    ("n_cold_user_val_interactions", cold_user_val_interactions),
    ("n_cold_user_test_interactions", cold_user_test_interactions),
    ("n_cold_item_val_interactions", cold_item_val_interactions),
    ("n_cold_item_test_interactions", cold_item_test_interactions),
    ("n_both_cold_val_interactions", both_cold_val_interactions),
    ("n_both_cold_test_interactions", both_cold_test_interactions),
]:
    user_meta = add_target_counts(user_meta, df, prefix)

for c in user_meta.columns:
    if c.startswith("n_") and pd.api.types.is_numeric_dtype(user_meta[c]):
        user_meta[c] = user_meta[c].fillna(0).astype("int32")

base_item_stats = pd.concat(
    [
        train_base.assign(is_positive=(train_base["rating"] >= POSITIVE_THRESHOLD).astype("uint8"))[["userId", "movieId", "is_positive"]],
        val_base.assign(is_positive=(val_base["rating"] >= POSITIVE_THRESHOLD).astype("uint8"))[["userId", "movieId", "is_positive"]],
        test_base.assign(is_positive=(test_base["rating"] >= POSITIVE_THRESHOLD).astype("uint8"))[["userId", "movieId", "is_positive"]],
    ],
    ignore_index=True,
)

item_meta = (
    base_item_stats.groupby("movieId")
    .agg(
        n_interactions_total=("userId", "size"),
        n_positive_total=("is_positive", "sum"),
        n_unique_users=("userId", "nunique"),
    )
    .reset_index()
)

item_meta["selected_cold_item"] = item_meta["movieId"].isin(sampled_cold_item_ids).astype("uint8")
item_meta["is_warm_item"] = item_meta["movieId"].isin(warm_item_ids).astype("uint8")
item_meta["is_cold_item"] = (1 - item_meta["is_warm_item"]).astype("uint8")
item_meta["item_split_group"] = np.where(item_meta["is_cold_item"] == 1, "cold", "warm")
item_meta["warm_item_idx"] = item_meta["movieId"].map(warm_item2idx).fillna(-1).astype("int32")
item_meta["cold_item_idx"] = item_meta["movieId"].map(cold_item2idx).fillna(-1).astype("int32")
item_meta["all_item_idx"] = item_meta["movieId"].map(all_item2idx).fillna(-1).astype("int32")

def add_item_presence(meta_df, movie_ids, col_name):
    unique_movie_ids = sorted(set(movie_ids))
    if not unique_movie_ids:
        meta_df[col_name] = 0
        meta_df[col_name] = meta_df[col_name].astype("uint8")
        return meta_df
    s = pd.Series(unique_movie_ids, dtype="int32", name="movieId").to_frame()
    s[col_name] = 1
    meta_df = meta_df.merge(s, on="movieId", how="left")
    meta_df[col_name] = meta_df[col_name].fillna(0).astype("uint8")
    return meta_df

item_meta = add_item_presence(item_meta, train_warm["movieId"].tolist(), "present_in_train_warm")
item_meta = add_item_presence(item_meta, warm_val_interactions["movieId"].tolist(), "present_in_warm_val_eval")
item_meta = add_item_presence(item_meta, warm_test_interactions["movieId"].tolist(), "present_in_warm_test_eval")
item_meta = add_item_presence(item_meta, cold_user_val_interactions["movieId"].tolist(), "present_in_cold_user_val_eval")
item_meta = add_item_presence(item_meta, cold_user_test_interactions["movieId"].tolist(), "present_in_cold_user_test_eval")
item_meta = add_item_presence(item_meta, cold_item_val_interactions["movieId"].tolist(), "present_in_cold_item_val_eval")
item_meta = add_item_presence(item_meta, cold_item_test_interactions["movieId"].tolist(), "present_in_cold_item_test_eval")
item_meta = add_item_presence(item_meta, both_cold_val_interactions["movieId"].tolist(), "present_in_both_cold_val_eval")
item_meta = add_item_presence(item_meta, both_cold_test_interactions["movieId"].tolist(), "present_in_both_cold_test_eval")

print_df_info("user_meta", user_meta)
print_df_info("item_meta", item_meta)


=== user_meta ===
shape: (159944, 27)
userId                            int64
n_interactions_total              int32
n_positive_total                  int32
n_train_interactions              int32
n_train_positive                  int32
n_val_interactions                int32
n_test_interactions               int32
n_val_positive_anchors            int32
n_test_positive_anchors           int32
n_val_positive_interactions       int32
n_test_positive_interactions      int32
first_val_timestamp               int64
first_test_timestamp              int64
last_test_timestamp               int64
is_cold_user                      uint8
is_warm_user                      uint8
warm_user_idx                     int32
cold_user_idx                     int32
user_split_group                 object
n_warm_val_interactions           int32
n_warm_test_interactions          int32
n_cold_user_val_interactions      int32
n_cold_user_test_interactions     int32
n_cold_item_val_interactions      int32
n

,userId,n_interactions_total,n_positive_total,n_train_interactions,n_train_positive,n_val_interactions,n_test_interactions,n_val_positive_anchors,n_test_positive_anchors,n_val_positive_interactions,n_test_positive_interactions,first_val_timestamp,first_test_timestamp,last_test_timestamp,is_cold_user,is_warm_user,warm_user_idx,cold_user_idx,user_split_group,n_warm_val_interactions,n_warm_test_interactions,n_cold_user_val_interactions,n_cold_user_test_interactions,n_cold_item_val_interactions,n_cold_item_test_interactions,n_both_cold_val_interactions,n_both_cold_test_interactions
0,1,70,39,61,34,3,6,2,3,2,3,1147879375,1147879603,1147880055,1,0,-1,0,cold,0,0,3,6,0,0,0,0
1,2,184,113,175,108,5,4,2,3,2,3,1141417831,1141417937,1141418008,0,1,0,-1,warm,5,4,0,0,0,0,0,0
2,3,656,357,643,352,7,6,2,3,2,3,1566091441,1566091612,1566091883,0,1,1,-1,warm,7,5,0,0,0,1,0,0



=== item_meta ===
shape: (32720, 20)
movieId                            int32
n_interactions_total               int64
n_positive_total                  uint64
n_unique_users                     int64
selected_cold_item                 uint8
is_warm_item                       uint8
is_cold_item                       uint8
item_split_group                  object
warm_item_idx                      int32
cold_item_idx                      int32
all_item_idx                       int32
present_in_train_warm              uint8
present_in_warm_val_eval           uint8
present_in_warm_test_eval          uint8
present_in_cold_user_val_eval      uint8
present_in_cold_user_test_eval     uint8
present_in_cold_item_val_eval      uint8
present_in_cold_item_test_eval     uint8
present_in_both_cold_val_eval      uint8
present_in_both_cold_test_eval     uint8
dtype: object


,movieId,n_interactions_total,n_positive_total,n_unique_users,selected_cold_item,is_warm_item,is_cold_item,item_split_group,warm_item_idx,cold_item_idx,all_item_idx,present_in_train_warm,present_in_warm_val_eval,present_in_warm_test_eval,present_in_cold_user_val_eval,present_in_cold_user_test_eval,present_in_cold_item_val_eval,present_in_cold_item_test_eval,present_in_both_cold_val_eval,present_in_both_cold_test_eval
0,1,57006,37640,57006,0,1,0,warm,0,-1,0,1,1,1,1,1,0,0,0,0
1,2,24150,8272,24150,0,1,0,warm,1,-1,1,1,1,1,1,1,0,0,0,0
2,3,11619,3661,11619,0,1,0,warm,2,-1,2,1,1,1,1,1,0,0,0,0


## 9. Построение итоговых item features

Формируется единая таблица признаков фильмов.  
Отдельно сохраняются признаки для всех фильмов и признаки только для warm-item пространства.


In [12]:
item_features_all = build_base_item_features(
    movies_df=movies,
    links_df=links,
    genome_scores_df=genome_scores,
    item_movie_ids=eligible_item_ids,
    genome_svd_dim=GENOME_SVD_DIM,
)

imdb_feat_df = build_imdb_item_features(
    links_df=links,
    catalog_movie_ids=eligible_item_ids,
    topk_thresholds=PERSON_TOPK_THRESHOLDS,
)

item_features_all = item_features_all.merge(
    imdb_feat_df.drop(columns=["imdbId"], errors="ignore"),
    on="movieId",
    how="left",
    suffixes=("", "_imdb"),
)

item_features_all = item_features_all.merge(
    item_meta[
        [
            "movieId",
            "all_item_idx",
            "warm_item_idx",
            "cold_item_idx",
            "is_warm_item",
            "is_cold_item",
            "selected_cold_item",
            "item_split_group",
        ]
    ],
    on="movieId",
    how="left",
)

for c in item_features_all.columns:
    if c in {"movieId", "title", "imdbId", "tmdbId", "tconst", "item_split_group"}:
        continue
    if pd.api.types.is_numeric_dtype(item_features_all[c]):
        item_features_all[c] = item_features_all[c].fillna(0)
    elif item_features_all[c].dtype == "bool":
        item_features_all[c] = item_features_all[c].astype("uint8")

item_features_all["all_item_idx"] = item_features_all["all_item_idx"].fillna(-1).astype("int32")
item_features_all["warm_item_idx"] = item_features_all["warm_item_idx"].fillna(-1).astype("int32")
item_features_all["cold_item_idx"] = item_features_all["cold_item_idx"].fillna(-1).astype("int32")
item_features_all["is_warm_item"] = item_features_all["is_warm_item"].fillna(0).astype("uint8")
item_features_all["is_cold_item"] = item_features_all["is_cold_item"].fillna(0).astype("uint8")
item_features_all["selected_cold_item"] = item_features_all["selected_cold_item"].fillna(0).astype("uint8")

item_features_all = item_features_all.sort_values("all_item_idx").reset_index(drop=True)
item_features_warm = item_features_all[item_features_all["is_warm_item"] == 1].copy().sort_values("warm_item_idx").reset_index(drop=True)

feature_cols = [c for c in item_features_all.columns if c not in NON_MODEL_ITEM_COLS]

print("Number of final model feature columns:", len(feature_cols))
print("First 60 feature columns:", feature_cols[:60])

print_df_info("item_features_all", item_features_all)
print_df_info("item_features_warm", item_features_warm)

Genome matrix shape=(13816, 1128), TruncatedSVD n_components=64
Number of final model feature columns: 160
First 60 feature columns: ['year', 'year_missing', 'decade', 'genre_count', 'genre__action', 'genre__adventure', 'genre__animation', 'genre__children', 'genre__comedy', 'genre__crime', 'genre__documentary', 'genre__drama', 'genre__fantasy', 'genre__film_noir', 'genre__horror', 'genre__imax', 'genre__musical', 'genre__mystery', 'genre__no_genres_listed', 'genre__romance', 'genre__sci_fi', 'genre__thriller', 'genre__war', 'genre__western', 'genome_missing', 'genome_svd_000', 'genome_svd_001', 'genome_svd_002', 'genome_svd_003', 'genome_svd_004', 'genome_svd_005', 'genome_svd_006', 'genome_svd_007', 'genome_svd_008', 'genome_svd_009', 'genome_svd_010', 'genome_svd_011', 'genome_svd_012', 'genome_svd_013', 'genome_svd_014', 'genome_svd_015', 'genome_svd_016', 'genome_svd_017', 'genome_svd_018', 'genome_svd_019', 'genome_svd_020', 'genome_svd_021', 'genome_svd_022', 'genome_svd_023', '

,movieId,title,year,year_missing,decade,imdbId,tmdbId,genre_count,genre__action,genre__adventure,genre__animation,genre__children,genre__comedy,genre__crime,genre__documentary,genre__drama,genre__fantasy,genre__film_noir,genre__horror,genre__imax,genre__musical,genre__mystery,genre__no_genres_listed,genre__romance,genre__sci_fi,genre__thriller,genre__war,genre__western,genome_missing,genome_svd_000,genome_svd_001,genome_svd_002,genome_svd_003,genome_svd_004,genome_svd_005,genome_svd_006,genome_svd_007,genome_svd_008,genome_svd_009,genome_svd_010,genome_svd_011,genome_svd_012,genome_svd_013,genome_svd_014,genome_svd_015,genome_svd_016,genome_svd_017,genome_svd_018,genome_svd_019,genome_svd_020,genome_svd_021,genome_svd_022,genome_svd_023,genome_svd_024,genome_svd_025,genome_svd_026,genome_svd_027,genome_svd_028,genome_svd_029,genome_svd_030,genome_svd_031,genome_svd_032,genome_svd_033,genome_svd_034,genome_svd_035,genome_svd_036,genome_svd_037,genome_svd_038,genome_svd_039,genome_svd_040,genome_svd_041,genome_svd_042,genome_svd_043,genome_svd_044,genome_svd_045,genome_svd_046,genome_svd_047,genome_svd_048,genome_svd_049,genome_svd_050,genome_svd_051,genome_svd_052,genome_svd_053,genome_svd_054,genome_svd_055,genome_svd_056,genome_svd_057,genome_svd_058,genome_svd_059,genome_svd_060,genome_svd_061,genome_svd_062,genome_svd_063,tconst,has_imdb_match,imdb_titleType_movie,imdb_titleType_short,imdb_titleType_tvEpisode,imdb_titleType_tvMiniSeries,imdb_titleType_tvMovie,imdb_titleType_tvSeries,imdb_titleType_tvShort,imdb_titleType_tvSpecial,imdb_titleType_video,isAdult,startYear,runtimeMinutes,runtimeMinutes_log1p,runtime_missing,imdb_genre__action,imdb_genre__adult,imdb_genre__adventure,imdb_genre__animation,imdb_genre__biography,imdb_genre__comedy,imdb_genre__crime,imdb_genre__documentary,imdb_genre__drama,imdb_genre__family,imdb_genre__fantasy,imdb_genre__film_noir,imdb_genre__history,imdb_genre__horror,imdb_genre__music,imdb_genre__musical,imdb_genre__mystery,imdb_genre__news,imdb_genre__reality_tv,imdb_genre__romance,imdb_genre__sci_fi,imdb_genre__short,imdb_genre__sport,imdb_genre__talk_show,imdb_genre__thriller,imdb_genre__war,imdb_genre__western,n_directors,n_writers,n_principals,cast_popularity_max,cast_popularity_mean,cast_popularity_sum,cast_top50_count,cast_top100_count,cast_top250_count,cast_top500_count,director_popularity_max,director_popularity_mean,director_popularity_sum,director_top50_count,director_top100_count,director_top250_count,director_top500_count,n_actor,n_actress,n_cast_total,n_distinct_principal_categories,cast_top50_share,director_top50_share,cast_top100_share,director_top100_share,cast_top250_share,director_top250_share,cast_top500_share,director_top500_share,all_item_idx,warm_item_idx,cold_item_idx,is_warm_item,is_cold_item,selected_cold_item,item_split_group
0,1,Toy Story (1995),1995.0,0,1990.0,114709,862,5,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,6.791893,1.134135,-2.296454,0.468671,1.481025,1.745133,0.681281,1.974400,-0.311326,0.820891,-1.232080,-0.191722,-0.596964,-0.785200,-0.871469,-0.694606,-0.978996,-0.131843,0.133362,0.138395,0.360950,-0.204245,-0.125046,0.673131,-0.434407,0.515651,0.192483,0.086528,-0.430010,0.257288,-0.099779,-0.043600,0.106876,-0.188670,0.089587,0.199802,0.489383,0.165422,-0.199824,-0.261710,-0.139076,0.019944,-0.185514,0.040400,0.125329,0.338982,0.077731,0.084698,0.044906,-0.085851,0.214432,-0.246713,-0.108180,0.104704,-0.049095,-0.106708,-0.241121,0.056145,-0.149804,0.310367,0.062833,-0.169643,0.091771,-0.175850,tt0114709,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1995.0,81.0,4.406719,0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,7,20,58.0,22.500000,225.0,0,1,2,2,7.0,7.0,7.0,0,0,0,0,8,2,10,4,0.0,0.0,0.100000,0.0,0.200000,0.0,0.200000,0.0,0,0,-1,1,0,0,warm
1,2,Jumanji (1995),1995.0,0,1990.0,113497,8844,3,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,4.962522,2.397015,-1.018293,0.029374,1.321550,0.216794,0.049347,1


=== item_features_warm ===
shape: (31999, 172)
movieId                             int32
title                      string[python]
year                              float32
year_missing                        uint8
decade                            float32
imdbId                              Int64
tmdbId                              Int64
genre_count                         int16
genre__action                       uint8
genre__adventure                    uint8
genre__animation                    uint8
genre__children                     uint8
genre__comedy                       uint8
genre__crime                        uint8
genre__documentary                  uint8
genre__drama                        uint8
genre__fantasy                      uint8
genre__film_noir                    uint8
genre__horror                       uint8
genre__imax                         uint8
genre__musical                      uint8
genre__mystery                      uint8
genre__no_genres_listed     

,movieId,title,year,year_missing,decade,imdbId,tmdbId,genre_count,genre__action,genre__adventure,genre__animation,genre__children,genre__comedy,genre__crime,genre__documentary,genre__drama,genre__fantasy,genre__film_noir,genre__horror,genre__imax,genre__musical,genre__mystery,genre__no_genres_listed,genre__romance,genre__sci_fi,genre__thriller,genre__war,genre__western,genome_missing,genome_svd_000,genome_svd_001,genome_svd_002,genome_svd_003,genome_svd_004,genome_svd_005,genome_svd_006,genome_svd_007,genome_svd_008,genome_svd_009,genome_svd_010,genome_svd_011,genome_svd_012,genome_svd_013,genome_svd_014,genome_svd_015,genome_svd_016,genome_svd_017,genome_svd_018,genome_svd_019,genome_svd_020,genome_svd_021,genome_svd_022,genome_svd_023,genome_svd_024,genome_svd_025,genome_svd_026,genome_svd_027,genome_svd_028,genome_svd_029,genome_svd_030,genome_svd_031,genome_svd_032,genome_svd_033,genome_svd_034,genome_svd_035,genome_svd_036,genome_svd_037,genome_svd_038,genome_svd_039,genome_svd_040,genome_svd_041,genome_svd_042,genome_svd_043,genome_svd_044,genome_svd_045,genome_svd_046,genome_svd_047,genome_svd_048,genome_svd_049,genome_svd_050,genome_svd_051,genome_svd_052,genome_svd_053,genome_svd_054,genome_svd_055,genome_svd_056,genome_svd_057,genome_svd_058,genome_svd_059,genome_svd_060,genome_svd_061,genome_svd_062,genome_svd_063,tconst,has_imdb_match,imdb_titleType_movie,imdb_titleType_short,imdb_titleType_tvEpisode,imdb_titleType_tvMiniSeries,imdb_titleType_tvMovie,imdb_titleType_tvSeries,imdb_titleType_tvShort,imdb_titleType_tvSpecial,imdb_titleType_video,isAdult,startYear,runtimeMinutes,runtimeMinutes_log1p,runtime_missing,imdb_genre__action,imdb_genre__adult,imdb_genre__adventure,imdb_genre__animation,imdb_genre__biography,imdb_genre__comedy,imdb_genre__crime,imdb_genre__documentary,imdb_genre__drama,imdb_genre__family,imdb_genre__fantasy,imdb_genre__film_noir,imdb_genre__history,imdb_genre__horror,imdb_genre__music,imdb_genre__musical,imdb_genre__mystery,imdb_genre__news,imdb_genre__reality_tv,imdb_genre__romance,imdb_genre__sci_fi,imdb_genre__short,imdb_genre__sport,imdb_genre__talk_show,imdb_genre__thriller,imdb_genre__war,imdb_genre__western,n_directors,n_writers,n_principals,cast_popularity_max,cast_popularity_mean,cast_popularity_sum,cast_top50_count,cast_top100_count,cast_top250_count,cast_top500_count,director_popularity_max,director_popularity_mean,director_popularity_sum,director_top50_count,director_top100_count,director_top250_count,director_top500_count,n_actor,n_actress,n_cast_total,n_distinct_principal_categories,cast_top50_share,director_top50_share,cast_top100_share,director_top100_share,cast_top250_share,director_top250_share,cast_top500_share,director_top500_share,all_item_idx,warm_item_idx,cold_item_idx,is_warm_item,is_cold_item,selected_cold_item,item_split_group
0,1,Toy Story (1995),1995.0,0,1990.0,114709,862,5,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,6.791893,1.134135,-2.296454,0.468671,1.481025,1.745133,0.681281,1.974400,-0.311326,0.820891,-1.232080,-0.191722,-0.596964,-0.785200,-0.871469,-0.694606,-0.978996,-0.131843,0.133362,0.138395,0.360950,-0.204245,-0.125046,0.673131,-0.434407,0.515651,0.192483,0.086528,-0.430010,0.257288,-0.099779,-0.043600,0.106876,-0.188670,0.089587,0.199802,0.489383,0.165422,-0.199824,-0.261710,-0.139076,0.019944,-0.185514,0.040400,0.125329,0.338982,0.077731,0.084698,0.044906,-0.085851,0.214432,-0.246713,-0.108180,0.104704,-0.049095,-0.106708,-0.241121,0.056145,-0.149804,0.310367,0.062833,-0.169643,0.091771,-0.175850,tt0114709,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1995.0,81.0,4.406719,0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,7,20,58.0,22.500000,225.0,0,1,2,2,7.0,7.0,7.0,0,0,0,0,8,2,10,4,0.0,0.0,0.100000,0.0,0.200000,0.0,0.200000,0.0,0,0,-1,1,0,0,warm
1,2,Jumanji (1995),1995.0,0,1990.0,113497,8844,3,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,4.962522,2.397015,-1.018293,0.029374,1.321550,0.216794,0.049347,1

## 10. Проверки корректности

Перед сохранением выполняются базовые проверки: корректность индексов, отсутствие пересечений в cold-item сценарии и соблюдение временного порядка.


In [13]:
assert train_warm["warm_user_idx"].ge(0).all()
assert train_warm["warm_item_idx"].ge(0).all()
assert train_warm["all_item_idx"].ge(0).all()

assert warm_val_interactions["warm_user_idx"].ge(0).all()
assert warm_val_interactions["warm_item_idx"].ge(0).all()

assert warm_test_interactions["warm_user_idx"].ge(0).all()
assert warm_test_interactions["warm_item_idx"].ge(0).all()

assert item_features_all["all_item_idx"].is_unique
assert set(item_features_all["movieId"]) == set(eligible_item_ids)

assert item_features_warm["warm_item_idx"].is_unique
assert set(item_features_warm["movieId"]) == set(warm_item_ids)

val_min_ts = warm_val_interactions.groupby("userId")["timestamp"].min().rename("val_min_ts")
train_max_ts = train_warm.groupby("userId")["timestamp"].max().rename("train_max_ts")
ts_check = train_max_ts.to_frame().join(val_min_ts, how="inner")
if len(ts_check):
    assert (ts_check["train_max_ts"] <= ts_check["val_min_ts"]).all()

if len(cold_user_support):
    assert cold_user_support["movieId"].isin(warm_item_ids).all()

if len(cold_item_val_interactions):
    assert (~cold_item_val_interactions["movieId"].isin(warm_item_ids)).all()
if len(cold_item_test_interactions):
    assert (~cold_item_test_interactions["movieId"].isin(warm_item_ids)).all()

print("All checks passed.")

All checks passed.


## 11. Сохранение артефактов

Сохраняются interaction-таблицы, признаки фильмов, meta-таблицы, mapping-файлы и конфигурация разбиения.


In [14]:
train_warm.to_parquet(OUT_DIR / "train_warm_interactions.parquet", index=False)

warm_val_interactions.to_parquet(OUT_DIR / "warm_val_interactions.parquet", index=False)
warm_test_interactions.to_parquet(OUT_DIR / "warm_test_interactions.parquet", index=False)

cold_user_support.to_parquet(OUT_DIR / "cold_user_support.parquet", index=False)
cold_user_support_all.to_parquet(OUT_DIR / "cold_user_support_all.parquet", index=False)

cold_user_val_interactions.to_parquet(OUT_DIR / "cold_user_val_interactions.parquet", index=False)
cold_user_test_interactions.to_parquet(OUT_DIR / "cold_user_test_interactions.parquet", index=False)

cold_item_val_interactions.to_parquet(OUT_DIR / "cold_item_val_interactions.parquet", index=False)
cold_item_test_interactions.to_parquet(OUT_DIR / "cold_item_test_interactions.parquet", index=False)

both_cold_val_interactions.to_parquet(OUT_DIR / "both_cold_val_interactions.parquet", index=False)
both_cold_test_interactions.to_parquet(OUT_DIR / "both_cold_test_interactions.parquet", index=False)

item_features_all.to_parquet(OUT_DIR / "item_features_all.parquet", index=False)
item_features_warm.to_parquet(OUT_DIR / "item_features_warm.parquet", index=False)

user_meta.to_parquet(OUT_DIR / "user_meta.parquet", index=False)
item_meta.to_parquet(OUT_DIR / "item_meta.parquet", index=False)

save_json({str(k): int(v) for k, v in warm_user2idx.items()}, OUT_DIR / "warm_user2idx.json")
save_json({str(k): int(v) for k, v in warm_item2idx.items()}, OUT_DIR / "warm_item2idx.json")
save_json({str(k): int(v) for k, v in all_item2idx.items()}, OUT_DIR / "all_item2idx.json")
save_json({str(k): int(v) for k, v in cold_user2idx.items()}, OUT_DIR / "cold_user2idx.json")
save_json({str(k): int(v) for k, v in cold_item2idx.items()}, OUT_DIR / "cold_item2idx.json")
save_json(feature_cols, OUT_DIR / "feature_cols.json")

split_config = {
    "seed": SEED,
    "positive_threshold": POSITIVE_THRESHOLD,
    "n_val_pos": N_VAL_POS,
    "n_test_pos": N_TEST_POS,
    "min_pos_per_user_for_split": MIN_POS_PER_USER_FOR_SPLIT,
    "min_user_interactions_kcore": MIN_USER_INTERACTIONS,
    "min_item_interactions_kcore": MIN_ITEM_INTERACTIONS,
    "cold_user_holdout": COLD_USER_HOLDOUT,
    "cold_item_holdout": COLD_ITEM_HOLDOUT,
    "min_support_interactions_for_cold_user": MIN_SUPPORT_INTERACTIONS_FOR_COLD_USER,
    "min_support_positives_for_cold_user": MIN_SUPPORT_POSITIVES_FOR_COLD_USER,
    "min_item_total_interactions_for_cold": MIN_ITEM_TOTAL_INTERACTIONS_FOR_COLD,
    "min_item_positive_interactions_for_cold": MIN_ITEM_POSITIVE_INTERACTIONS_FOR_COLD,
    "min_item_unique_users_for_cold": MIN_ITEM_UNIQUE_USERS_FOR_COLD,
    "genome_svd_dim": GENOME_SVD_DIM,
    "person_topk_thresholds": list(PERSON_TOPK_THRESHOLDS),
    "principal_categories": list(PRINCIPAL_CATEGORIES),
    "n_users_total": int(len(eligible_user_ids)),
    "n_items_total": int(len(eligible_item_ids)),
    "n_users_warm": int(len(warm_user2idx)),
    "n_users_cold": int(len(cold_user2idx)),
    "n_items_warm": int(len(warm_item2idx)),
    "n_items_cold": int(len(cold_item2idx)),
    "n_train_warm_interactions": int(len(train_warm)),
    "n_warm_val_interactions": int(len(warm_val_interactions)),
    "n_warm_test_interactions": int(len(warm_test_interactions)),
    "n_cold_user_val_interactions": int(len(cold_user_val_interactions)),
    "n_cold_user_test_interactions": int(len(cold_user_test_interactions)),
    "n_cold_item_val_interactions": int(len(cold_item_val_interactions)),
    "n_cold_item_test_interactions": int(len(cold_item_test_interactions)),
    "n_both_cold_val_interactions": int(len(both_cold_val_interactions)),
    "n_both_cold_test_interactions": int(len(both_cold_test_interactions)),
    "n_feature_cols": int(len(feature_cols)),
}
save_json(split_config, OUT_DIR / "split_config.json")

print("Saved to:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print(f"{p.name:40s} {p.stat().st_size / 1024**2:10.2f} MB")

Saved to: /kaggle/working/processed_ml25m_cold_eval_all_ratings_v2
all_item2idx.json                              0.56 MB
both_cold_test_interactions.parquet            0.13 MB
both_cold_val_interactions.parquet             0.08 MB
cold_item2idx.json                             0.01 MB
cold_item_test_interactions.parquet            1.15 MB
cold_item_val_interactions.parquet             0.76 MB
cold_user2idx.json                             0.27 MB
cold_user_support.parquet                     21.63 MB
cold_user_support_all.parquet                 23.45 MB
cold_user_test_interactions.parquet            1.65 MB
cold_user_val_interactions.parquet             1.11 MB
feature_cols.json                              0.00 MB
item_features_all.parquet                      8.56 MB
item_features_warm.parquet                     8.14 MB
item_meta.parquet                              0.78 MB
split_config.json                              0.00 MB
train_warm_interactions.parquet              192.71 M

## 12. Финальная сводка

Короткая проверка размеров итогового датасета и числа признаков.  
Эти значения удобно сверять перед запуском EDA и экспериментов.


In [15]:
print("=== FINAL SUMMARY ===")
print("Users total:", len(eligible_user_ids))
print("Items total:", len(eligible_item_ids))
print()
print("Warm users:", len(warm_user2idx))
print("Cold users:", len(cold_user2idx))
print("Warm items:", len(warm_item2idx))
print("Cold items:", len(cold_item2idx))
print()
print("Train warm interactions:", len(train_warm))
print()
print("Warm val/test interactions:", len(warm_val_interactions), "/", len(warm_test_interactions))
print("Cold-user val/test interactions:", len(cold_user_val_interactions), "/", len(cold_user_test_interactions))
print("Cold-item val/test interactions:", len(cold_item_val_interactions), "/", len(cold_item_test_interactions))
print("Both-cold val/test interactions:", len(both_cold_val_interactions), "/", len(both_cold_test_interactions))
print()
print("Cold-user support (warm items):", len(cold_user_support))
print("Cold-user support all:", len(cold_user_support_all))
print()
print("Item feature columns:", len(feature_cols))
print(feature_cols[:80])

=== FINAL SUMMARY ===
Users total: 159944
Items total: 32720

Warm users: 144009
Cold users: 15935
Warm items: 31999
Cold items: 721

Train warm interactions: 19574307

Warm val/test interactions: 604994 / 987722
Cold-user val/test interactions: 66243 / 106919
Cold-item val/test interactions: 35162 / 55411
Both-cold val/test interactions: 3820 / 6060

Cold-user support (warm items): 2170334
Cold-user support all: 2295457

Item feature columns: 160
['year', 'year_missing', 'decade', 'genre_count', 'genre__action', 'genre__adventure', 'genre__animation', 'genre__children', 'genre__comedy', 'genre__crime', 'genre__documentary', 'genre__drama', 'genre__fantasy', 'genre__film_noir', 'genre__horror', 'genre__imax', 'genre__musical', 'genre__mystery', 'genre__no_genres_listed', 'genre__romance', 'genre__sci_fi', 'genre__thriller', 'genre__war', 'genre__western', 'genome_missing', 'genome_svd_000', 'genome_svd_001', 'genome_svd_002', 'genome_svd_003', 'genome_svd_004', 'genome_svd_005', 'genom

---

### Примечание

Сформированные файлы можно загружать в последующих ноутбуках через директорию `data/movielens20-withfeatures-split/` или через соответствующий Kaggle input path.  
Семантика положительных/отрицательных оценок намеренно не фиксируется на этапе подготовки датасета.
